In [15]:
import pandas as pd
armas1 = pd.read_csv(r'armas1_processed.csv')
armas2 = pd.read_csv(r'armas2_processed.csv')
armas3 = pd.read_csv(r'armas3_processed.csv')
combined_df = pd.concat([armas1, armas2, armas3], ignore_index=True)
# Optional: Handle missing data (NaN values) if necessary
combined_df = combined_df.fillna('No Data')
# Save the combined DataFrame to a new CSV file
output_filename = 'armas_all_processed.csv'
combined_df.to_csv(output_filename, index=False)
armas_all =pd.read_csv(r'armas_all_processed.csv')

# import libraries

In [16]:
!pip install scikit-learn

import pandas as pd
import matplotlib.pyplot as plt
import sklearn
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline       
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.inspection import permutation_importance


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


# Copy your train,validation and test sets

In [18]:
train_df = armas1.copy()
valid_df = armas2.copy()    
test_df  = armas3.copy()

# Define the target and columns to remove

In [20]:
TARGET = 'Index_K'

DROP_COLS = [TARGET, 'NAIRASV3', 'NAIRASV2', 'Datetime', 'Vehicle_ID', 'ARMAS']

# Separate input features X and output y

In [24]:
def split_xy(df):
    X = df.drop(columns=DROP_COLS, errors='ignore')
    y = df[TARGET]
    return X, y
X_train, y_train = split_xy(train_df)
X_valid, y_valid = split_xy(valid_df)
X_test, y_test   = split_xy(test_df)

In [48]:
print("Train shape:", X_train.shape)
print("Valid shape:", X_valid.shape)
print("Test shape :", X_test.shape)

print("\nTrain class counts:")
print(y_train.value_counts().sort_index())

Train shape: (32380, 43)
Valid shape: (29309, 43)
Test shape : (30787, 43)

Train class counts:
Index_K
0     8871
1    18185
2     5324
Name: count, dtype: int64


Detect the numeric and categorical columns

In [30]:
numeric_cols = X_train.select_dtypes(include=['number']).columns.tolist()
categorical_cols = X_train.select_dtypes(exclude=['number']).columns.tolist()

print("\nNumeric columns:", numeric_cols)
print("Categorical columns:", categorical_cols)


Numeric columns: ['Latitude', 'Longitude', 'Altitude(Bar)', 'Altitude(GPS)', 'Geomagnetic_latitude', 'Geomagnetic_longitude', 'Geomagnetic_Rc', 'Geomagnetic_Lshell', 'NM_NEWK', 'NM_OULU', 'NM_THUL', 'NM_SOPO', 'SXR_short', 'SXR_long', 'Particles_P1', 'Particles_P5', 'Particles_P10', 'Particles_P30', 'Particles_P50', 'Particles_P100', 'Particles_E20', 'SW_B', 'SW_Bx', 'SW_By', 'SW_Bz', 'SW_V', 'SW_Vx', 'SW_Vy', 'SW_Vz', 'SW_density', 'SW_temperature', 'SW_pressure', 'Index_Kp', 'Index_Dst', 'Index_Ap', 'Solar_sunspots', 'Solar_f107', 'Solar_NPF', 'Solar_SPF', 'Solar_APF', 'Solar_NPF20', 'Solar_SPF20', 'Solar_APF20']
Categorical columns: []


Build preprocessing pipeline

In [31]:
numeric_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer([
    ('num', numeric_pipe, numeric_cols),
    ('cat', categorical_pipe, categorical_cols)
])

Manual Tuining of logistic regression

In [35]:
datasets = {
    "armas1": armas1,
    "armas2": armas2,
    "armas3": armas3
}

In [37]:
rotations = [
    ("armas1", "armas2", "armas3"),
    ("armas1", "armas3", "armas2"),
    ("armas2", "armas1", "armas3"),
    ("armas2", "armas3", "armas1"),
    ("armas3", "armas1", "armas2"),
    ("armas3", "armas2", "armas1")
]

In [ ]:
#------------------------------------k neighbors-classifier with tuning---------

from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import cross_val_score
from sklearn.inspection import permutation_importance

# Preprocess the training and validation data
X_train_processed = preprocessor.fit_transform(X_train)
X_valid_processed = preprocessor.transform(X_valid)
X_test_processed = preprocessor.transform(X_test)

print("="*70)
print("KNN CLASSIFIER - OPTIMIZED WITH n_neighbors=12")
print("="*70)

# ===== STEP 1: Feature Selection using Permutation Importance =====
print("\n1. FEATURE IMPORTANCE ANALYSIS")
print("-" * 70)

# Train a baseline model for importance analysis
baseline_knn = KNeighborsClassifier(n_neighbors=12, weights='distance', metric='euclidean')
baseline_knn.fit(X_train_processed, y_train)

# Calculate permutation importance
perm_importance = permutation_importance(baseline_knn, X_valid_processed, y_valid, n_repeats=10, random_state=42)
feature_importance_df = pd.DataFrame({
    'importance': perm_importance.importances_mean,
    'std': perm_importance.importances_std
}).sort_values('importance', ascending=False)

print(f"Top 10 Important Features:")
print(feature_importance_df.head(10))

# Keep features with positive importance (threshold > 0)
important_features = feature_importance_df[feature_importance_df['importance'] > 0].index.tolist()
print(f"\nNumber of important features: {len(important_features)} out of {X_train_processed.shape[1]}")

X_train_selected = X_train_processed[:, important_features]
X_valid_selected = X_valid_processed[:, important_features]
X_test_selected = X_test_processed[:, important_features]

# ===== STEP 2: Compare Different Metrics and Parameters =====
print("\n2. HYPERPARAMETER TUNING - Comparing Different Metrics")
print("-" * 70)

metrics_to_test = ['euclidean', 'manhattan', 'minkowski']
results = []

for metric in metrics_to_test:
    knn = KNeighborsClassifier(n_neighbors=12, weights='distance', metric=metric)
    
    # Cross-validation on selected features
    cv_scores = cross_val_score(knn, X_train_selected, y_train, cv=5, scoring='accuracy')
    
    # Train and evaluate on validation set
    knn.fit(X_train_selected, y_train)
    y_pred_valid = knn.predict(X_valid_selected)
    valid_acc = accuracy_score(y_valid, y_pred_valid)
    
    results.append({
        'Metric': metric,
        'CV Accuracy': cv_scores.mean(),
        'CV Std': cv_scores.std(),
        'Validation Accuracy': valid_acc
    })
    
    print(f"\n{metric.upper()}:")
    print(f"  Cross-Validation Accuracy: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")
    print(f"  Validation Accuracy:       {valid_acc:.4f}")

# ===== STEP 3: Select Best Model and Test on Test Set =====
best_result = max(results, key=lambda x: x['Validation Accuracy'])
best_metric = best_result['Metric']

print("\n3. FINAL MODEL EVALUATION")
print("-" * 70)
print(f"Best Metric: {best_metric.upper()}")
print(f"Best Validation Accuracy: {best_result['Validation Accuracy']:.4f}")

# Train final model with best metric
final_knn = KNeighborsClassifier(n_neighbors=12, weights='distance', metric=best_metric)
final_knn.fit(X_train_selected, y_train)

# Validation set evaluation
y_pred_valid_final = final_knn.predict(X_valid_selected)
print(f"\nValidation Set Results:")
print(f"Accuracy: {accuracy_score(y_valid, y_pred_valid_final):.4f}")
print(f"\nClassification Report (Validation):")
print(classification_report(y_valid, y_pred_valid_final))

# Test set evaluation
y_pred_test = final_knn.predict(X_test_selected)
print(f"\nTest Set Results:")
print(f"Accuracy: {accuracy_score(y_test, y_pred_test):.4f}")
print(f"\nClassification Report (Test):")
print(classification_report(y_test, y_pred_test))

# Sample prediction
sample = X_train.iloc[[0]]
sample_processed = preprocessor.transform(sample)
sample_selected = sample_processed[:, important_features]
print(f"\nSample Prediction (first training sample):")
print(f"Prediction: {final_knn.predict(sample_selected)}")
print(f"Probability: {final_knn.predict_proba(sample_selected)}")

KNN CLASSIFIER - OPTIMIZED WITH n_neighbors=12

1. FEATURE IMPORTANCE ANALYSIS
----------------------------------------------------------------------
